# Style2Fit — Image Generation Pipeline
AIPI 540 Mini Hackathon #3  
Run on Google Colab with A100 GPU runtime.  
Project files are in Google Drive: `MyDrive/GAI_hackathon/Style2Fit/`

In [ ]:
# Mount Google Drive and set working directory
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/GAI_hackathon/Style2Fit"
os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")
print(f"Files: {os.listdir('.')}")

In [ ]:
# Install dependencies
!pip install -q diffusers transformers accelerate safetensors pillow

In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Load Stable Diffusion XL
from diffusers import StableDiffusionXLPipeline
import torch

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    variant="fp16",
)
pipe = pipe.to("cuda")
print("SDXL loaded successfully")

In [ ]:
# Load text results from training notebook (already on Drive)
import json

with open("demo_data_text.json", "r") as f:
    demo_data = json.load(f)

print(f"Loaded {len(demo_data)} demo entries")
for entry in demo_data:
    print(f"  - {entry['id']}: {entry['prompt'][:50]}...")

In [ ]:
# Generate before/after images for each demo prompt
import os
from PIL import Image

os.makedirs("images_before", exist_ok=True)
os.makedirs("images_after", exist_ok=True)

NEGATIVE_PROMPT = "blurry, low quality, distorted face, extra limbs, bad anatomy, watermark, text, logo, cropped, worst quality"

def get_subject(gender):
    if gender == "female":
        return "a young woman"
    elif gender == "male":
        return "a young man"
    else:
        return "a person"

def make_image_prompt(subject, outfit_text):
    return f"Full body fashion photograph, {subject} wearing {outfit_text}, natural lighting, clean background, fashion editorial style, high quality, detailed clothing textures, 8k"

generator = torch.Generator(device="cuda").manual_seed(42)

for i, entry in enumerate(demo_data):
    print(f"\n[{i+1}/{len(demo_data)}] Generating images for: {entry['id']}")
    subject = get_subject(entry["gender"])

    # Before image
    before_prompt = make_image_prompt(subject, entry["before_text"][:200])
    generator = torch.Generator(device="cuda").manual_seed(42)
    before_img = pipe(
        prompt=before_prompt,
        negative_prompt=NEGATIVE_PROMPT,
        num_inference_steps=30,
        generator=generator,
    ).images[0]
    before_img.save(f"images_before/{entry['id']}.png")
    print(f"  \u2713 Before image saved")

    # After image
    after_prompt = make_image_prompt(subject, entry["after_text"][:200])
    generator = torch.Generator(device="cuda").manual_seed(42)
    after_img = pipe(
        prompt=after_prompt,
        negative_prompt=NEGATIVE_PROMPT,
        num_inference_steps=30,
        generator=generator,
    ).images[0]
    after_img.save(f"images_after/{entry['id']}.png")
    print(f"  \u2713 After image saved")

print("\nAll images generated!")

In [ ]:
# Convert images to base64 and create final demo_data.json
import base64
from pathlib import Path

final_data = []
for entry in demo_data:
    new_entry = dict(entry)

    # Read before image
    before_path = f"images_before/{entry['id']}.png"
    with open(before_path, "rb") as f:
        before_b64 = base64.b64encode(f.read()).decode("utf-8")
    new_entry["before_image_base64"] = f"data:image/png;base64,{before_b64}"

    # Read after image
    after_path = f"images_after/{entry['id']}.png"
    with open(after_path, "rb") as f:
        after_b64 = base64.b64encode(f.read()).decode("utf-8")
    new_entry["after_image_base64"] = f"data:image/png;base64,{after_b64}"

    final_data.append(new_entry)

with open("demo_data.json", "w") as f:
    json.dump(final_data, f)

file_size = Path("demo_data.json").stat().st_size / 1e6
print(f"Saved demo_data.json ({file_size:.1f} MB) to Drive")
print("Find it at: MyDrive/GAI_hackathon/Style2Fit/demo_data.json")
print("Use this file to update DEMO_DATA in the demo app.")

In [ ]:
# Display before/after grid for visual verification
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, len(demo_data), figsize=(4 * len(demo_data), 8))

for i, entry in enumerate(demo_data):
    before_img = Image.open(f"images_before/{entry['id']}.png")
    after_img = Image.open(f"images_after/{entry['id']}.png")

    axes[0, i].imshow(before_img)
    axes[0, i].set_title(f"Before\n{entry['id']}", fontsize=8)
    axes[0, i].axis("off")

    axes[1, i].imshow(after_img)
    axes[1, i].set_title(f"After\n{entry['id']}", fontsize=8)
    axes[1, i].axis("off")

plt.suptitle("Style2Fit \u2014 Before vs After Image Comparison", fontsize=14)
plt.tight_layout()
plt.savefig("comparison_grid.png", dpi=150, bbox_inches="tight")
plt.show()
print("Grid saved to comparison_grid.png")